# CareerQuest: fine-tune the career retrieval embedder

**Goal.** Given a job posting or résumé-like text, retrieve the right O*NET occupation. We fine-tune a small sentence embedder on pairs built from O*NET alternate titles and LinkedIn postings, then compare Recall@5 before and after on postings the model never saw.

**Run this on a GPU** (Colab: Runtime → Change runtime type → T4 GPU, or a Kaggle notebook with GPU on). Training takes roughly 10 to 20 minutes.

**Inputs.** From `product/backend`, run `python scripts/build_finetune_pairs.py`. Upload the files it writes to `data/processed/finetune/`: `corpus.jsonl`, `train.jsonl`, `eval.jsonl`.

**Licenses.** O*NET 31.0 is CC BY 4.0. The pairs and the fine-tuned model are derived from LinkedIn Job Postings (arshkon, CC BY-SA 4.0), so share them only under CC BY-SA 4.0 with credit, and keep them inside the class project.

In [ ]:
!pip install -q "sentence-transformers>=3.3,<4" "datasets>=2.19"

In [ ]:
import json, os, pathlib, torch

BASE_MODEL = "BAAI/bge-small-en-v1.5"   # 33M parameters, fast, strong baseline
# Kaggle sets KAGGLE_KERNEL_RUN_TYPE; Colab also has a read-only /kaggle/input, so don't test for that.
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
DATA = pathlib.Path("/kaggle/input/careerquest-finetune") if ON_KAGGLE else pathlib.Path("finetune")
if not ON_KAGGLE:
    DATA.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training will be slow)")

try:  # On Colab, pick the three .jsonl files when prompted.
    from google.colab import files
    for name, content in files.upload().items():
        (DATA / name).write_bytes(content)
except ImportError:
    pass  # Kaggle: add the files as a dataset named careerquest-finetune.
print(sorted(p.name for p in DATA.glob("*.jsonl")))

In [ ]:
def load(name):
    with open(DATA / name, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

corpus, train, held_out = load("corpus.jsonl"), load("train.jsonl"), load("eval.jsonl")
print(f"{len(corpus):,} occupations, {len(train):,} training pairs, {len(held_out):,} held-out postings")
print("Training pairs by source:", {s: sum(r['source'] == s for r in train) for s in {r['source'] for r in train}})

## Baseline: the base model before fine-tuning
`accuracy@5` is Recall@5 here, because each posting has exactly one right occupation.

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

evaluator = InformationRetrievalEvaluator(
    queries={str(i): row["query"] for i, row in enumerate(held_out)},
    corpus={doc["soc"]: doc["text"] for doc in corpus},
    relevant_docs={str(i): {row["soc"]} for i, row in enumerate(held_out)},
    accuracy_at_k=[1, 5, 10], precision_recall_at_k=[5], mrr_at_k=[10], ndcg_at_k=[10],
    name="careerquest", show_progress_bar=False,
)

def summary(results):
    return {k.split("_", 1)[1]: round(v, 4) for k, v in results.items()
            if any(m in k for m in ("accuracy@1", "accuracy@5", "accuracy@10", "mrr@10"))}

model = SentenceTransformer(BASE_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
before = summary(evaluator(model))
before

## Fine-tune with in-batch negatives
`MultipleNegativesRankingLoss` treats the other occupations in each batch as negatives. The `NO_DUPLICATES` sampler stops two pairs for the same occupation from sharing a batch, which would otherwise teach the model that an occupation is a wrong answer for itself.

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers

train_dataset = Dataset.from_dict({"anchor": [r["query"] for r in train], "positive": [r["positive"] for r in train]})
args = SentenceTransformerTrainingArguments(
    output_dir="careerquest-embedder-checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    seed=13,
)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_dataset,
                                     loss=losses.MultipleNegativesRankingLoss(model))
trainer.train()

In [ ]:
after = summary(evaluator(model))
import pandas as pd
table = pd.DataFrame({"base": before, "fine-tuned": after})
table["change"] = table["fine-tuned"] - table["base"]
table

## Save and download
If Recall@5 did not improve, keep the base model: it is still a valid result to report. Try 2 epochs or a lower learning rate (1e-5) before concluding.

In [ ]:
model.save_pretrained("careerquest-embedder")
with open("careerquest-embedder/metrics.json", "w") as handle:
    json.dump({"base_model": BASE_MODEL, "before": before, "after": after,
               "train_pairs": len(train), "eval_queries": len(held_out),
               "license": "Derived from LinkedIn Job Postings (arshkon, CC BY-SA 4.0) and O*NET 31.0 (CC BY 4.0)"}, handle, indent=2)
!zip -qr careerquest-embedder.zip careerquest-embedder
try:
    from google.colab import files
    files.download("careerquest-embedder.zip")
except ImportError:
    print("On Kaggle, download careerquest-embedder.zip from the Output panel.")

## Use it in CareerQuest
1. Unzip so the model sits at `product/backend/models/embedder/` (that folder holds `config.json`, `model.safetensors` and `metrics.json`).
2. From `product/backend`: `python -m pip install sentence-transformers`.
3. Check it on your machine: `python scripts/eval_retrieval.py --model models/embedder`, and compare with `python scripts/eval_retrieval.py` (TF-IDF).
4. Restart the API. Résumé career suggestions then use the embedder; without the folder they fall back to TF-IDF.